# Um mapa a partir de uma tabela: monumentos da Bairrada

Seminário de Investigação 2026/27 · Universidade de Coimbra

Correr cada célula com **Shift+Enter**. A tabela `locais.csv` tem sete monumentos e lugares históricos da Bairrada (Mealhada, Anadia, Cantanhede, Oliveira do Bairro), em quatro colunas: `nome`, `tipo`, `lat`, `lon`. Para usar os vossos dados, substituam o ficheiro (menu *File → Upload*) mantendo as mesmas colunas.

## 0. Instalar os pacotes (só demora se faltarem)

In [ ]:
%pip install -q pandas matplotlib contextily folium pypdf

## 1. Ler a tabela

In [ ]:
import pandas as pd

locais = pd.read_csv("locais.csv")
locais

## 2. Mapa com matplotlib e fundo de mapa (Esri)

O `contextily` descarrega o mapa de fundo (ruas, rio, edifícios) e coloca os pontos por cima. As coordenadas são convertidas para a projeção dos mapas web (Web Mercator).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx

# latitude/longitude -> Web Mercator (metros), a projeção dos mapas web
R = 6378137
locais["x"] = R * np.radians(locais.lon)
locais["y"] = R * np.log(np.tan(np.pi / 4 + np.radians(locais.lat) / 2))

def desenhar(fundo, titulo):
    fig, ax = plt.subplots(figsize=(9, 9))
    for tipo, grupo in locais.groupby("tipo"):
        ax.scatter(grupo.x, grupo.y, s=120, label=tipo, edgecolor="white", zorder=3)
    for _, l in locais.iterrows():
        ax.annotate(l.nome, (l.x, l.y), xytext=(6, 6), textcoords="offset points",
                    fontsize=9, zorder=4,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, lw=0))
    margem = 2500  # metros à volta dos pontos
    ax.set_xlim(locais.x.min() - margem, locais.x.max() + 4 * margem)  # mais espaço à direita para os nomes
    ax.set_ylim(locais.y.min() - margem, locais.y.max() + margem)
    ctx.add_basemap(ax, crs="EPSG:3857", source=fundo)  # fundo Esri, sem chave de acesso
    ax.set_axis_off()
    ax.legend(loc="lower right")
    ax.set_title(titulo)
    plt.show()

desenhar(ctx.providers.Esri.WorldStreetMap, "Bairrada: monumentos e lugares históricos")

## 2b. O mesmo mapa com imagem de satélite

Basta trocar o fundo: `Esri.WorldImagery` é imagem de satélite, gratuita e sem chave de acesso, o tipo de vista usado na investigação em fontes abertas (OSINT).

In [ ]:
desenhar(ctx.providers.Esri.WorldImagery, "Bairrada: monumentos sobre imagem de satélite")

## 3. O mesmo, como mapa interativo (Folium)

No canto superior direito escolhe-se o fundo: ruas ou satélite. Clicar num marcador mostra o nome e o tipo.

In [ ]:
import folium

cores = {"Palácio": "red", "Termas": "blue", "Museu": "green", "Religioso": "purple"}

mapa = folium.Map(location=[locais.lat.mean(), locais.lon.mean()], zoom_start=11, tiles=None)
folium.TileLayer("Esri.WorldStreetMap", name="Ruas").add_to(mapa)
folium.TileLayer("Esri.WorldImagery", name="Satélite").add_to(mapa)
for _, l in locais.iterrows():
    folium.Marker(
        [l.lat, l.lon],
        popup=f"<b>{l.nome}</b><br>{l.tipo}",
        tooltip=l.nome,
        icon=folium.Icon(color=cores.get(l.tipo, "gray")),
    ).add_to(mapa)
folium.LayerControl().add_to(mapa)   # botão para trocar ruas / satélite

mapa.save("mapa.html")   # para descarregar e abrir no navegador
mapa